# Sewage Defect Detection — Tier 2 Training on SageMaker (GPU)

Run YOLO26m with the **P2 small-object head** at imgsz=640 with the
augmentation / loss-weight recipe that the local M1 Pro run could not
apply (the older Ultralytics on Mac silently dropped `cls_weights`).

**Target instance**: GPU notebook — pick one of

| Instance | GPU | VRAM | Approx $/hr | Suggested batch |
|---|---|---:|---:|---:|
| ml.g4dn.xlarge | NVIDIA T4 | 16 GB | $0.74 | 8 |
| ml.g5.xlarge | NVIDIA A10G | 24 GB | $1.20 | 16 |
| ml.g6.xlarge | NVIDIA L4 | 24 GB | $0.80 | 16 |
| ml.g5.2xlarge | NVIDIA A10G | 24 GB | $1.50 | 16 (more CPU) |

A 200-epoch run on the 686-image train split should land in ~1.5 h on
ml.g5.xlarge, ~2.5 h on ml.g4dn.xlarge.

Works on both **SageMaker Studio** and **SageMaker Notebook Instances**
(detected automatically). Conda env: `pytorch_p310` or any kernel with
PyTorch ≥ 2.1 and CUDA.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, platform, torch
print()
print('python  :', platform.python_version())
print('torch   :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('VRAM    :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GiB')
is_studio = os.path.exists('/home/sagemaker-user')
WORKSPACE = '/home/sagemaker-user/cnn-assignment3' if is_studio else '/home/ec2-user/SageMaker/cnn-assignment3'
print('workspace:', WORKSPACE)

## 2. Install dependencies

Uses SageMaker's existing CUDA-enabled PyTorch; only adds the project-specific extras.

In [ ]:
!pip install -q -U \
    'ultralytics>=8.4.51' \
    'sahi>=0.11.36' \
    'pycocotools>=2.0.11' \
    'onnx>=1.17' 'onnxslim>=0.1.71' 'onnxruntime' \
    boto3 sagemaker

## 3. Stage the dataset

Two paths — pick **one** of the next two cells.

- **3a**: dataset is already in S3 (recommended for repeatable runs).
- **3b**: zip the local dataset, upload via Studio's UI, then unzip here.

### 3a — Pull from S3

Expected layout in your bucket: `s3://<bucket>/sewage-yolo26/{train,valid,test}/{images,labels}/` and `data.yaml`.

In [ ]:
import os, subprocess
S3_DATASET_URI = 's3://YOUR_BUCKET/sewage-yolo26/'  # EDIT ME
os.makedirs(f'{WORKSPACE}/src/data', exist_ok=True)
target = f'{WORKSPACE}/src/data/sewage-yolo26'
if not os.path.exists(target):
    subprocess.run(['aws', 's3', 'sync', S3_DATASET_URI, target, '--quiet'], check=True)
    print('Synced from', S3_DATASET_URI)
else:
    print('Already present:', target)
!ls {target}

### 3b — Upload a zip via Studio, then unzip

Make a zip locally: `zip -r sewage-yolo26.zip src/data/sewage-yolo26`, drop it into the Studio file browser, then run:

In [ ]:
import os, zipfile
ZIP_PATH = 'sewage-yolo26.zip'   # path you uploaded to
TARGET   = f'{WORKSPACE}/src/data/'
os.makedirs(TARGET, exist_ok=True)
if not os.path.exists(f'{TARGET}/sewage-yolo26'):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(TARGET)
print(os.listdir(f'{TARGET}/sewage-yolo26'))

## 4. Normalise dataset paths

Roboflow exports use relative paths like `../train/images` that resolve
outside the yaml folder on some setups. Rewrite to absolute form.

In [ ]:
import yaml
data_yaml = f'{WORKSPACE}/src/data/sewage-yolo26/data.yaml'
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path']  = f'{WORKSPACE}/src/data/sewage-yolo26'
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'
with open(data_yaml, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(data_yaml).read())

## 5. Tier 2 training

- **Architecture**: `yolo26m-p2.yaml` — adds stride-4 P2 detection head for small defects.
- **imgsz=640** — matches the Roboflow export resolution (1280 would only interpolate).
- **Augmentation**: mosaic=1.0, close_mosaic=20, mixup=0.10, copy_paste=0.30, scale=0.6.
- **Optimizer**: MuSGD (default — CUDA supports its bf16 path fine).
- **Schedule**: cosine LR, 200 epochs, patience=50.
- **Class weights**: enabled (requires ultralytics ≥ 8.4.51).

In [ ]:
from ultralytics import YOLO

# Build P2-head architecture, transfer compatible weights from yolo26m.
model = YOLO('yolo26m-p2.yaml')
try:
    model.load('yolo26m.pt')
    print('Loaded yolo26m.pt pretrained weights into P2 architecture.')
except Exception as exc:
    print('Pretrained transfer failed:', exc, '— training from scratch.')

# Class importance weights aligned with data.yaml order.
CIW_MAP = {
    'Buckling': 1.8, 'Crack': 2.6, 'Debris': 1.0,
    'Hole': 2.4, 'Joint offset': 1.7, 'Obstacle': 1.2, 'Utility intrusion': 2.8,
}
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
cls_weights = [CIW_MAP.get(n, 1.0) for n in names]
print('cls_weights:', dict(zip(names, cls_weights)))

# Pick a batch size based on detected VRAM.
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 16 if vram_gb >= 20 else 8        # A10G/L4 → 16, T4 → 8
print(f'using batch={BATCH} for {vram_gb:.0f} GiB GPU')

train_kwargs = dict(
    data=data_yaml,
    epochs=200,
    imgsz=640,
    batch=BATCH,
    device=0,
    optimizer='MuSGD',
    cos_lr=True,
    patience=50,
    plots=True,
    project='runs/detect',
    name='tier2_p2_sagemaker',
    # Augmentation
    mosaic=1.0, close_mosaic=20,
    mixup=0.10, copy_paste=0.30,
    scale=0.6, degrees=10.0, translate=0.10,
    hsv_v=0.30, hsv_s=0.60, fliplr=0.5,
    # Class weighting (8.4.51+)
    cls_weights=cls_weights,
    # Performance
    amp=True,
    workers=4,
)
results = model.train(**train_kwargs)

## 6. Evaluate on the test split — per-class metrics

In [ ]:
BEST = 'runs/detect/tier2_p2_sagemaker/weights/best.pt'
best = YOLO(BEST)
test_metrics = best.val(
    data=data_yaml, split='test', imgsz=640, device=0,
    conf=0.001, iou=0.6, augment=True, plots=True,
    project='runs/detect', name='tier2_p2_test', exist_ok=True,
)
print()
print(f'TEST | mAP@0.5 = {test_metrics.box.map50:.4f}')
print(f'TEST | mAP@0.5:0.95 = {test_metrics.box.map:.4f}')
print(f'TEST | P = {test_metrics.box.mp:.4f}')
print(f'TEST | R = {test_metrics.box.mr:.4f}')

## 7. SAHI slice-size sweep (optional)

Skip this on the SageMaker run if you already confirmed locally that
SAHI does not help on the 640 px Roboflow split. Run it only if you
ingest higher-resolution data later.

In [ ]:
RUN_SAHI = False
if RUN_SAHI:
    import sys, importlib, os
    sys.path.insert(0, f'{WORKSPACE}/src')
    os.chdir(WORKSPACE)
    import sahi_inference
    importlib.reload(sahi_inference)
    sahi_inference.WEIGHTS_PATH = BEST
    sahi_inference.EVAL_SPLIT = 'test'
    sahi_inference.SLICE_SIZE_SWEEP_ENABLE = True
    sahi_inference.SLICE_SIZE_SWEEP_CANDIDATES = (256, 384, 512, 640)
    sahi_inference.RUN_STANDARD_BASELINE = True
    sahi_inference.main()
else:
    print('Skipped (set RUN_SAHI=True to enable).')

## 8. Export to ONNX and TensorRT

TensorRT export needs CUDA, so SageMaker is the natural home for it.
We also produce ONNX for cross-platform serving.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, dynamic=True, simplify=True)
print('ONNX:', onnx_path)

try:
    engine_path = best.export(
        format='engine',
        imgsz=640,
        half=True,
        dynamic=True,
        workspace=4,
    )
    print('TensorRT:', engine_path)
except Exception as exc:
    print('TensorRT export failed (often needs the matching TRT version on the instance):', exc)

## 9. Persist artifacts to S3

SageMaker instances are ephemeral — anything not pushed to S3 is lost when the kernel/instance shuts down.

In [ ]:
import subprocess, time
S3_OUTPUT_URI = 's3://YOUR_BUCKET/runs/'  # EDIT ME
stamp = time.strftime('%Y%m%d-%H%M')
dest = f'{S3_OUTPUT_URI.rstrip("/")}/tier2_p2_sagemaker_{stamp}/'
subprocess.run(
    ['aws', 's3', 'sync', 'runs/detect/tier2_p2_sagemaker', dest, '--quiet'],
    check=True,
)
subprocess.run(
    ['aws', 's3', 'sync', 'runs/detect/tier2_p2_test', dest + 'test_eval/', '--quiet'],
    check=True,
)
print('Saved to', dest)

## 10. Shut down the instance

SageMaker Notebook Instances keep billing until manually stopped. After
the artifacts are in S3, stop the instance from the AWS console or via:

In [ ]:
# Uncomment only when you are done. This kills the kernel.
# !sudo shutdown -h now